In [1]:
from pathlib import Path
from google.colab import drive
import hashlib, io, json, os, shutil, subprocess, sys, zipfile
from datetime import datetime, timezone

drive.mount('/content/drive')
PERSIST = Path('/content/drive/MyDrive/Thesis/UniFashion/week8')
WORK = Path('/content/unifashion_completion')
RUN = PERSIST / 'evaluations/completion_fp32_step90000'
TRAIN = PERSIST / 'runs/full_train_fp32_b2_seed42'
BASE_CHECKPOINT = PERSIST / 'tuning/fp32_lr_batch_01/parent_snapshot.pt'
ATTENTION_RESULTS = None  # Only set after teammate supplies full common-protocol evaluation results.
WORK.mkdir(parents=True, exist_ok=True)
RUN.mkdir(parents=True, exist_ok=True)

def extract_checked(archive, root):
    for member in archive.infolist():
        if not (root / member.filename).resolve().is_relative_to(root.resolve()):
            raise ValueError('Unsafe ZIP member: ' + member.filename)
    archive.extractall(root)

bundle = PERSIST / 'inputs/unifashion_completion.zip'
assert bundle.is_file(), f'Upload the new completion bundle: {bundle}'
with zipfile.ZipFile(bundle) as z:
    manifest = json.loads(z.read('completion_manifest.json'))
    assert set(z.namelist()) == set(manifest['files']) | {'completion_manifest.json'}
    for name, expected in manifest['files'].items():
        assert hashlib.sha256(z.read(name)).hexdigest() == expected, name
    reference = json.loads(z.read('completion_inputs/base/unifashion_image_caption_score.json'))
    fingerprint = reference['provenance']['training_fingerprint']
    with zipfile.ZipFile(io.BytesIO(z.read('completion_inputs/original_code.zip'))) as original:
        for name, expected in fingerprint['code'].items():
            paths = [p for p in original.namelist() if Path(p).name == name]
            assert len(paths) == 1 and hashlib.sha256(original.read(paths[0])).hexdigest() == expected, name
        extract_checked(original, WORK)
    protected = set(fingerprint['code'])
    assert not any(Path(p).name in protected for p in manifest['files']), 'Addon would replace original training code'
    extract_checked(z, WORK)

with zipfile.ZipFile(PERSIST / 'inputs/unifashion_week8_data.zip') as z:
    assert hashlib.sha256(z.read('data/data_audit.json')).hexdigest() == fingerprint['data_audit']
    extract_checked(z, WORK)

assets = PERSIST / 'pretrained_assets'
assert (assets / 'download_manifest.json').is_file(), 'Keep original pretrained_assets on Drive'
(WORK / 'checkpoints').mkdir(exist_ok=True)
link = WORK / 'checkpoints/unifashion'
if not link.exists():
    link.symlink_to(assets, target_is_directory=True)
assert link.resolve() == assets.resolve()
os.chdir(WORK)
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
os.environ['HF_HOME'] = str(WORK / '.cache/huggingface')
INPUTS = WORK / 'completion_inputs'
SNAPSHOT = RUN / 'fp32_step90000.pt'
RESULTS = RUN / 'final_results'
REPORT = RUN / 'report'

def run_live(command):
    with subprocess.Popen([str(x) for x in command], cwd=WORK, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise RuntimeError(f'Command failed: exit {process.returncode}; see output above')

assert (TRAIN / 'last.pt').is_file()
assert BASE_CHECKPOINT.is_file(), 'Keep the evaluated step60000 parent snapshot'
print('Original training:', TRAIN)
print('Persistent evaluation/report:', RUN)
print('Attention-fusion: pending unless ATTENTION_RESULTS is supplied')


Mounted at /content/drive
Original training: /content/drive/MyDrive/Thesis/UniFashion/week8/runs/full_train_fp32_b2_seed42
Persistent evaluation/report: /content/drive/MyDrive/Thesis/UniFashion/week8/evaluations/completion_fp32_step90000
Attention-fusion: pending unless ATTENTION_RESULTS is supplied


In [2]:
if not ((3, 10) <= sys.version_info[:2] <= (3, 12)):
    raise RuntimeError('Use Python 3.10–3.12 for this pinned environment')
run_live([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-unifashion.txt'])
run_live([sys.executable, '-B', 'scripts/check_unifashion_environment.py'])
run_live([sys.executable, '-B', '-m', 'pytest', 'tests/test_fashioniq_completion.py',
          'tests/test_fashioniq_evaluation.py::test_official_inference_single_query_matches_batch_and_caption_score', '-q',
          '--junitxml=' + str(RUN / 'completion_tests.xml')])
run_live([sys.executable, '-c',
          'import torch; assert torch.cuda.is_available(), "Select a GPU runtime"; print(torch.cuda.get_device_name())'])


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 9.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 10.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 21.0 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 10.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB

In [3]:
run_live([sys.executable, '-B', 'scripts/finalize_unifashion_checkpoint.py', 'snapshot',
          '--source', TRAIN / 'last.pt', '--output', SNAPSHOT,
          '--reference-result', INPUTS / 'base/unifashion_image_caption_score.json'])
for name in ('status.json', 'config.json', 'environment.json', 'epoch_metrics.csv', 'training_curves.png'):
    path = TRAIN / name
    if path.exists():
        shutil.copyfile(path, RUN / ('training_' + name))


{
  "source": "/content/drive/MyDrive/Thesis/UniFashion/week8/runs/full_train_fp32_b2_seed42/last.pt",
  "checkpoint": "/content/drive/MyDrive/Thesis/UniFashion/week8/evaluations/completion_fp32_step90000/fp32_step90000.pt",
  "checkpoint_sha256": "1b41d1f442785030c51148b0538097b17fd1aec8b3d7653c0b4829adf448b9e5",
  "optimizer_steps": 90000,
  "completed_epochs": 10,
  "cursor": [
    10,
    0
  ],
  "nonvision_tensors": 351,
  "copy_sha256_verified": true,
  "cpu_deserialization_verified": true,
  "note": "Full model state loading is checked by GPU evaluation; this receipt verifies copying and CPU deserialization."
}


In [4]:
# Keep pandas/NumPy imports in fresh processes after the dependency install.

status_file = RESULTS / 'status.json'
done = status_file.exists() and json.loads(status_file.read_text()).get('status') == 'evaluation_completed'
if done:
    verify_code = """
import json, sys
from pathlib import Path
from src.fashioniq_completion import read_result
results, eval_data, receipt = map(Path, sys.argv[1:])
snapshot_info = json.loads(receipt.read_text())
for method in ('unifashion_image_score', 'unifashion_image_caption_score'):
    checked = read_result(results, method, eval_data, 'original', 90000)
    if checked['provenance']['checkpoint_sha256'] != snapshot_info['checkpoint_sha256']:
        raise ValueError('Completed results belong to a different checkpoint')
"""
    run_live([sys.executable, '-B', '-c', verify_code, RESULTS,
              INPUTS / 'evaluation_data', SNAPSHOT.with_suffix('.json')])
    print('Complete results verified; skipping GPU rerun.')
else:
    if RESULTS.exists():
        archived = RESULTS.with_name('final_results_interrupted_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
        RESULTS.rename(archived)
        print('Preserved interrupted results:', archived)
    run_live([sys.executable, '-B', '-u', 'scripts/evaluate_unifashion_final.py',
              '--checkpoint', SNAPSHOT, '--reference-result', INPUTS / 'base/unifashion_image_caption_score.json',
              '--eval-data', INPUTS / 'evaluation_data', '--data-dir', WORK / 'data',
              '--images-dir', WORK / 'images', '--output', RESULTS, '--batch-size', '4'])


Preserved interrupted results: /content/drive/MyDrive/Thesis/UniFashion/week8/evaluations/completion_fp32_step90000/final_results_interrupted_20261102T094500333237Z
Restored recorded source line endings: /content/unifashion_completion/third_party/UniFashion/src/blip_fine_tune_2.py
Restored recorded source line endings: /content/unifashion_completion/third_party/UniFashion/src/data_utils.py
Restored recorded source line endings: /content/unifashion_completion/third_party/UniFashion/src/lavis/common/registry.py
Restored recorded source line endings: /content/unifashion_completion/third_party/UniFashion/src/lavis/common/dist_utils.py
Restored recorded source line endings: /content/unifashion_completion/third_party/UniFashion/src/lavis/common/utils.py
Restored recorded source line endings: /content/unifashion_completion/third_party/UniFashion/src/lavis/common/logger.py
Restored recorded source line endings: /content/unifashion_completion/third_party/UniFashion/src/lavis/models/base_model.p

In [5]:
def generate_report():
    command = [sys.executable, '-B', 'scripts/report_unifashion_completion.py',
               '--inputs', INPUTS, '--output', REPORT]
    if (RESULTS / 'status.json').exists() and json.loads((RESULTS / 'status.json').read_text()).get('status') == 'evaluation_completed':
        command += ['--final-evaluation', RESULTS]
    if ATTENTION_RESULTS is not None:
        command += ['--attention-evaluation', Path(ATTENTION_RESULTS)]
    run_live(command)

generate_report()
from IPython.display import display, Markdown
display(Markdown((REPORT / 'weekly_report.md').read_text()))
print('Selection:', (REPORT / 'checkpoint_selection.json').read_text())


{
  "original_training_completed": true,
  "original_final_evaluated": true,
  "selected_checkpoint_copy_receipt_valid": false,
  "attention_fusion_evaluated": false,
  "comparison_split": "val",
  "official_test_recall_available": false,
  "test_blocker": "6118 public test queries have no target labels in the supplied data",
  "last_week_task_completed": false,
  "three_model_validation_comparison_completed": false,
  "original_test_task_completed": false
}


# Báo cáo gộp tuần 43–44: hoàn tất UniFashion và chuẩn bị so sánh mô hình

Nhiệm vụ: (1) hoàn tất huấn luyện và lưu checkpoint tốt nhất; (2) so sánh baseline additive, UniFashion fine-tune và attention-fusion bằng Recall@10/@50.

**Phạm vi hiện tại là validation, không phải test chính thức.** Dữ liệu local có 6.118 test query nhưng toàn bộ target bị ẩn; chưa thể tính test Recall. Cần thống nhất phạm vi với giảng viên hoặc có cơ chế đánh giá test chính thức. Không đổi tên validation thành test.

## 1. Minh chứng hoàn tất huấn luyện

ZIP do người chạy cung cấp xác nhận run gốc `full_train_fp32_b2_seed42` hoàn tất **10/10 epoch, 90.000 optimizer update, 180.000 lượt ví dụ**. Mỗi epoch có 18.000 query, batch 2, 9.000 update và không bỏ mẫu. Đã kiểm tra thứ tự step/batch, các giá trị hữu hạn, tính lại loss từng epoch và đối chiếu fingerprint với run gốc đã đánh giá tại step 60.000.

Cấu hình: FP32, seed 42, AdamW max LR 2e-5, OneCycle 10 epoch, EVA-G đóng băng; Tesla T4, PyTorch 2.6.0+cu124. Monitor là 96 query cố định, khác full validation 6.016 query.

| Epoch | Step | Train loss | Monitor loss |
|---:|---:|---:|---:|
| 1 | 9000 | 0.303700 | 0.356786 |
| 2 | 18000 | 0.366083 | 0.770699 |
| 3 | 27000 | 0.330310 | 0.993188 |
| 4 | 36000 | 0.295580 | 0.582520 |
| 5 | 45000 | 0.263618 | 0.263252 |
| 6 | 54000 | 0.202468 | 0.257138 |
| 7 | 63000 | 0.182144 | 0.223193 |
| 8 | 72000 | 0.131991 | 0.266174 |
| 9 | 81000 | 0.106700 | 0.254015 |
| 10 | 90000 | 0.120989 | 0.266318 |

![Đường cong training từ run gốc](../training_training_curves.png)

Monitor loss thấp nhất: **0.223193 ở epoch 7**. `best_monitor.pt` được chọn theo loss; không suy ra đây là checkpoint tốt nhất theo Recall. Train loss giảm so với giai đoạn đầu nhưng tăng nhẹ từ epoch 9 sang epoch 10; chưa có cơ sở kết luận hội tụ hay chất lượng truy xuất của checkpoint cuối từ loss.

Tổng thời gian các update được ghi là 29.89 giờ; không phải tổng GPU/wall-clock cost vì chưa gồm setup, validation, lưu checkpoint và các bước chạy lại. XML trong ZIP ghi nhận 16 tests đã đạt tại môi trường Colab; đây là kết quả kiểm thử được xuất, không phải một lượt GPU test mới tại máy local.

## 2. Lựa chọn checkpoint theo Recall

| Checkpoint gốc | Macro R@10 (%) | Macro R@50 (%) | Mean R (%) |
|---|---:|---:|---:|
| step60000 | 62.57 | 79.18 | 70.88 |
| step90000 | 64.79 | 80.64 | 72.71 |

Checkpoint được chọn trong các ứng viên đã đo: **step90000**. Tiêu chí cố định: mean macro R@10/R@50 với score caption gallery; hòa thì giữ step60000. Không đưa attention-fusion vào cùng quyết định vì đó là mô hình khác.

**Chưa có receipt hợp lệ chứng minh lưu/nạp lại bản best riêng.** ZIP training không chứa trọng số; việc ghi `Saved checkpoint` trong log không thay thế kiểm tra file.

## 3. So sánh các mô hình trên full validation

Cùng 6.016 query: dress 2.017/3.817 gallery, shirt 2.038/6.346, toptee 1.961/5.373. Loại reference khỏi ranking, cùng raw text và gallery caption cho các biến thể UniFashion; macro là trung bình đều ba category. Baseline được tái sử dụng từ kết quả đã đo và kiểm tra lại rank; không tuyên bố chạy inference baseline mới.

| Mô hình / checkpoint | Category | R@10 (%) | R@50 (%) |
|---|---|---:|---:|
| FashionCLIP additive | dress | 21.91 | 41.79 |
| FashionCLIP additive | shirt | 22.13 | 37.63 |
| FashionCLIP additive | toptee | 26.82 | 45.18 |
| FashionCLIP additive | MACRO | 23.62 | 41.54 |
| UniFashion step60000 + caption | dress | 62.37 | 78.48 |
| UniFashion step60000 + caption | shirt | 59.62 | 76.30 |
| UniFashion step60000 + caption | toptee | 65.73 | 82.76 |
| UniFashion step60000 + caption | MACRO | 62.57 | 79.18 |
| UniFashion step60000 no caption score | dress | 34.46 | 59.54 |
| UniFashion step60000 no caption score | shirt | 43.13 | 64.38 |
| UniFashion step60000 no caption score | toptee | 45.89 | 70.27 |
| UniFashion step60000 no caption score | MACRO | 41.16 | 64.73 |
| UniFashion step90000 + caption | dress | 65.20 | 80.66 |
| UniFashion step90000 + caption | shirt | 61.24 | 77.58 |
| UniFashion step90000 + caption | toptee | 67.92 | 83.68 |
| UniFashion step90000 + caption | MACRO | 64.79 | 80.64 |
| UniFashion step90000 no caption score | dress | 39.46 | 65.34 |
| UniFashion step90000 no caption score | shirt | 48.09 | 67.27 |
| UniFashion step90000 no caption score | toptee | 47.88 | 72.97 |
| UniFashion step90000 no caption score | MACRO | 45.14 | 68.53 |

**Attention-fusion: chưa có kết quả cùng giao thức.** Theo cập nhật mới nhất của người chạy, teammate chưa hoàn tất run dự định dùng. Không lấy điểm của gate FashionCLIP độc lập hoặc notebook cũ để điền vào hàng UniFashion + attention-fusion. Bổ sung checkpoint và full-validation JSON/CSV khi sẵn sàng.

## 4. Giới hạn và phần còn lại

- Huấn luyện gốc đã hoàn tất; không cần train lại hoặc chạy lại tuning.
- Đã đo checkpoint cuối.
- Cần lưu bản best riêng, xác minh checksum và nạp lại.
- Chờ teammate hoàn tất và đánh giá attention-fusion.
- Validation đã dùng lựa chọn checkpoint, không phải test độc lập. Baseline cache chưa được đối chiếu hoàn toàn pixel ảnh nguồn với run hiện tại.
- Điểm cao hơn paper về số học không chứng minh vượt paper vì protocol/caption/preprocessing khác nhau.
- Minh chứng kiểm tra log/rank không thay thế chạy lại mô hình độc lập.

## 5. Đoạn tóm tắt báo cáo

Nhóm đã hoàn tất huấn luyện UniFashion trên FashionIQ với 10 epoch và 90.000 bước cập nhật ở cấu hình FP32, batch size 2. Minh chứng xuất từ Colab được kiểm tra tính liên tục và nhất quán với các checkpoint trước. Monitor loss thấp nhất đạt 0,223193 tại epoch 7, trong khi checkpoint được đánh giá tốt nhất trước đó tại step 60.000 đạt macro Recall@10/Recall@50 là 62,57%/79,18%. Sau đánh giá checkpoint cuối, nhóm chọn step90000 theo mean macro Recall. Kết quả attention-fusion sẽ được bổ sung sau khi teammate hoàn tất run. Các kết quả hiện có thuộc tập validation; chưa báo cáo test Recall do dữ liệu test không có nhãn target.


Selection: {
  "criterion": "mean full-validation macro Recall@10 and Recall@50",
  "scoring_method": "unifashion_image_caption_score",
  "tie_rule": "retain step60000",
  "selected": {
    "name": "step90000",
    "optimizer_steps": 90000,
    "Recall@10": 64.78562335993237,
    "Recall@50": 80.64073431929766,
    "mean_R": 72.71317883961501,
    "checkpoint_sha256": "1b41d1f442785030c51148b0538097b17fd1aec8b3d7653c0b4829adf448b9e5",
    "training_fingerprint": {
      "data_audit": "4715235209dc1b8573217a9bc97c1cc41f76d8cc29e66f2a6b56bb9f4ae21bbe",
      "code": {
        "train_unifashion_finetune.py": "0c8b1887b00865b00bdfc0b54db745551e19a18f003fab9c59a65fbd0096c09c",
        "unifashion_training.py": "01ba71bc80e0fba5af64f5bba31a16cef7cf54042a5ef013a26ab97e4b745525",
        "unifashion_runtime.py": "ab27b7634b40bf271957f67311a775bc7d976184311676dcd957eda875ceed66",
        "unifashion_sanity_data.py": "3e2c8b0d47d6632f38526360fce8bf57c111fdbc02f0e26033687e9fe5fad06b",
        "fa

In [6]:
run_live([sys.executable, '-B', 'scripts/finalize_unifashion_checkpoint.py', 'save-best',
          '--selection', REPORT / 'checkpoint_selection.json', '--base', BASE_CHECKPOINT,
          '--final', SNAPSHOT, '--output-dir', PERSIST / 'selected_checkpoints/completion'])
generate_report()  # Updates the report with the verified copy receipt.
print((REPORT / 'checkpoint_copy.json').read_text())


{
  "source": "/content/drive/MyDrive/Thesis/UniFashion/week8/evaluations/completion_fp32_step90000/fp32_step90000.pt",
  "checkpoint": "/content/drive/MyDrive/Thesis/UniFashion/week8/selected_checkpoints/completion/best_recall_step90000.pt",
  "checkpoint_sha256": "1b41d1f442785030c51148b0538097b17fd1aec8b3d7653c0b4829adf448b9e5",
  "optimizer_steps": 90000,
  "completed_epochs": 10,
  "cursor": [
    10,
    0
  ],
  "nonvision_tensors": 351,
  "copy_sha256_verified": true,
  "cpu_deserialization_verified": true,
  "note": "Full model state loading is checked by GPU evaluation; this receipt verifies copying and CPU deserialization.",
  "selection": {
    "criterion": "mean full-validation macro Recall@10 and Recall@50",
    "scoring_method": "unifashion_image_caption_score",
    "tie_rule": "retain step60000",
    "selected": {
      "name": "step90000",
      "optimizer_steps": 90000,
      "Recall@10": 64.78562335993237,
      "Recall@50": 80.64073431929766,
      "mean_R": 72.7131

In [7]:
evidence_zip = RUN / 'completion_evidence.zip'
allowed = {'.json', '.jsonl', '.csv', '.log', '.png', '.md', '.xml'}
with zipfile.ZipFile(evidence_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN.rglob('*')):
        if path.is_file() and path.suffix in allowed and not any('_interrupted_' in p for p in path.relative_to(RUN).parts):
            archive.write(path, path.relative_to(RUN).as_posix())
print('Send this ZIP:', evidence_zip)
print('Keep checkpoint files on Drive; they are not in the ZIP.')
from google.colab import files
files.download(str(evidence_zip))


Send this ZIP: /content/drive/MyDrive/Thesis/UniFashion/week8/evaluations/completion_fp32_step90000/completion_evidence.zip
Keep checkpoint files on Drive; they are not in the ZIP.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>